[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/05_04_main_tfidf.ipynb)

# Module 5, NLP: TF-IDF. A Lexical Map of a Corpus

**Notebook:** `05_04_main_tfidf`

## What we're doing

In the embeddings notebook, we represented each research abstract with a learned semantic vector.

Here we use the **exact same frozen OpenAlex corpus**, but change the representation.

> **Same corpus/data. Different representation, and a different notion of similarity.**

With TF-IDF, documents are similar when they use similar **distinctive words and phrases**.

With embeddings, documents can be similar because they express similar **meanings**, even when they use different vocabulary.

That gives us a useful progression:

- classical NLP: **we choose the features**;
- TF-IDF: **the corpus vocabulary becomes the feature space**;
- embeddings: **a model learns the representation**.

## What we'll do

1. Load the fixed OpenAlex corpus.
2. Clean the text for lexical analysis.
3. Build a sparse TF-IDF matrix.
4. Inspect one document's strongest features.
5. Build a "more like this" engine with cosine similarity.
6. Cluster the corpus with KMeans.
7. Interpret clusters using centroid terms and representative papers.
8. Visualize the space with TruncatedSVD.
9. Experiment with the number of clusters.
10. Compare TF-IDF with embeddings.

## The key question

> **What structure can we recover when similarity is based on shared vocabulary rather than learned semantic meaning?**

## 0) Setup

This notebook uses standard scientific-Python tools available in Google Colab. No GPU or external language model is required.

In [ ]:
import re
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD

print("Setup complete.")

## 1) Load the same frozen OpenAlex corpus

The embeddings notebook showed how the corpus can be collected from the OpenAlex API. For analysis, we use a **fixed copy** stored in GitHub so every student works with the same observations.

The source file is:

```text
nlp_data/openalex_ml_corpus.csv
```

Holding the corpus constant lets us isolate the effect of the representation:

> Are differences in the results coming from the documents, or from how we represented them?

In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "tunnel-ai/way/main/nlp_data/openalex_ml_corpus.csv"
)

try:
    df = pd.read_csv(DATA_URL)
except Exception as e:
    raise RuntimeError(
        "Could not load openalex_ml_corpus.csv. "
        "Make sure it has been added to tunnel-ai/way/nlp_data."
    ) from e

required = {"title", "abstract"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")

df = df[df["abstract"].notna()].copy()
df["abstract"] = df["abstract"].astype(str)
df["title"] = df["title"].fillna("").astype(str)

print(f"Loaded {len(df):,} papers with usable abstracts.")
df.head(3)

### A reminder about the corpus

The OpenAlex walkthrough used a filtered, citation-sorted sample of recent machine-learning papers. It is useful for learning NLP methods, but it is **not a census of machine-learning research**.

Cluster sizes and themes therefore describe **this sampled corpus**.

Our methodological question is narrower:

> What does TF-IDF reveal about the structure of these documents?

## 2) Light cleaning for TF-IDF

TF-IDF works directly with tokens, so cleaning decisions matter.

We will:

- lowercase;
- remove URLs;
- remove most punctuation while keeping letters, numbers, and hyphens;
- normalize whitespace.

We are not stemming or lemmatizing, which keeps the resulting features easy to inspect.

> Cleaning is an argument about what information you consider irrelevant. It is never completely neutral.

In [ ]:
url_pat = re.compile(r"https?://\S+|www\.\S+")
multi_space_pat = re.compile(r"\s+")

def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = url_pat.sub(" ", s.strip()).lower()
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    return multi_space_pat.sub(" ", s).strip()

df["text"] = df["abstract"].apply(clean_text)

example_i = 0
print("TITLE:", df.iloc[example_i]["title"])
print("\nRAW:\n", textwrap.shorten(
    df.iloc[example_i]["abstract"], width=350, placeholder="…"
))
print("\nCLEAN:\n", textwrap.shorten(
    df.iloc[example_i]["text"], width=350, placeholder="…"
))

## 3) Represent every abstract as a TF-IDF vector

**TF-IDF** stands for **term frequency–inverse document frequency**.

A term receives more weight when:

- it appears frequently in one document;
- but appears in relatively few documents across the corpus.

Conceptually:

\[
\text{abstract} \longrightarrow [x_1, x_2, \ldots, x_V]
\]

where \(V\) is the retained vocabulary.

Unlike embeddings, the dimensions are human-readable words and phrases.

### Choices we make

- `stop_words="english"` removes common English stop words; (is this a good idea?)
- `min_df=5` removes very rare terms;
- `max_df=0.90` removes near-universal terms;
- `ngram_range=(1, 2)` keeps single words and two-word phrases;
- `max_features=15000` caps the vocabulary.

These are modeling choices, not universal truths.

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    min_df=5,
    max_df=0.90,
    ngram_range=(1, 2),
    max_features=15_000,
    sublinear_tf=True,
)

X = vectorizer.fit_transform(df["text"])
feature_names = vectorizer.get_feature_names_out()

print("TF-IDF matrix:", X.shape)
print(f"Documents: {X.shape[0]:,}")
print(f"Vocabulary features: {X.shape[1]:,}")
print(f"Non-zero entries: {X.nnz:,}")
print(f"Matrix density: {X.nnz / (X.shape[0] * X.shape[1]):.4%}")

### Sparse is the point

Most documents use only a small fraction of the corpus vocabulary, so almost all entries in the TF-IDF matrix are zero.

| Representation | Typical shape | Meaning of dimensions | Storage |
| --- | ---: | --- | --- |
| TF-IDF | documents × thousands of terms | explicit words/phrases | sparse |
| Embeddings | documents × 384 | learned latent dimensions | dense |

Neither representation is automatically better. They preserve different information.

## 4) What does TF-IDF think is distinctive about one paper?

One of TF-IDF's biggest advantages is transparency: we can inspect the vector and see which words or phrases received the largest weights.

In [ ]:
def show_top_terms(doc_i, top_n=12):
    row = X[doc_i]

    if row.nnz == 0:
        print("This document has no retained TF-IDF features.")
        return

    order = np.argsort(row.data)[::-1][:top_n]
    term_indices = row.indices[order]
    weights = row.data[order]

    print("TITLE:", df.iloc[doc_i]["title"], "\n")
    for j, w in zip(term_indices, weights):
        print(f"  {feature_names[j]:<32s} {w:.3f}")

show_top_terms(0)

Read the output like a lexical fingerprint.

Ask:

- Do the top terms describe the paper well?
- Are any features surprising?
- What aspects of meaning are missing when the representation only knows about vocabulary?

## 5) Find similar papers with cosine similarity

Once every abstract is a vector, we can compare vectors.

**Cosine similarity** measures how closely two vectors point in the same direction.

For TF-IDF, high similarity usually means the documents share similar patterns of distinctive vocabulary.

This gives us a simple **"more like this"** engine—the same basic primitive used in document retrieval and recommendation.

### Why use a focused example?

TF-IDF is strongest when related documents share distinctive vocabulary.  
For the demonstration below, we use the paper **HaluEval**, which focuses on hallucination evaluation in large language models.

If TF-IDF is working as expected, nearby papers should tend to discuss related ideas such as hallucination detection, LLM evaluation, prompting, or self-refinement.

That makes this a useful sanity check before we ask TF-IDF to organize the entire corpus.

In [ ]:
def show_neighbors(query_idx, k=8):
    sim = cosine_similarity(X[query_idx], X).ravel()
    order = np.argsort(sim)[::-1]

    print("Query:")
    print(" •", df.iloc[query_idx]["title"])
    print("\nNearest neighbors:")

    shown = 0
    for j in order:
        if j == query_idx:
            continue

        title = textwrap.shorten(
            str(df.iloc[j]["title"]),
            width=95,
            placeholder="…",
        )

        print(f"  sim={sim[j]:.3f} | {title}")
        shown += 1

        if shown >= k:
            break


# Use a focused example where lexical overlap should work well.
matches = df.index[df["title"].str.contains("HaluEval", case=False, na=False)].tolist()

if matches:
    query_idx = matches[0]
    show_neighbors(query_idx)
else:
    print("HaluEval was not found in the corpus; using the first document instead.")
    show_neighbors(0)

### A data-quality check hiding inside the results

If you see a similarity score very close to `1.0`, inspect the titles. You may have found the same paper indexed twice, different versions of the same work, or near-duplicate abstracts.

That is not necessarily a model failure.

It may be the method correctly revealing a **data-quality issue**.

> Model output reflects both the analytical method and the data we feed it.

## 6) Cluster the TF-IDF vectors with KMeans

We now move from pairwise similarity to corpus structure.

Start with:

```python
K = 10
```

As in the embeddings notebook, this is an analytical choice—not a claim that machine learning has exactly ten objectively correct subfields.

In [ ]:
K = 10

kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=10,
)

df["cluster"] = kmeans.fit_predict(X)

print("Cluster sizes:")
print(df["cluster"].value_counts().sort_index())

## 7) Interpret the clusters using centroid terms

A KMeans centroid represents the average location of a cluster in the TF-IDF feature space.

Because TF-IDF dimensions correspond to explicit vocabulary, we can inspect the centroid's largest weights.

Those terms give us evidence for interpreting the group.

They do **not** automatically name a true subfield.

In [ ]:
TOP_N = 10
cluster_top_terms = {}

for c in range(K):
    centroid = kmeans.cluster_centers_[c]
    top_idx = centroid.argsort()[::-1][:TOP_N]
    cluster_top_terms[c] = [feature_names[j] for j in top_idx]

print("Top centroid terms by cluster:\n")

for c in range(K):
    n = int((df["cluster"] == c).sum())
    terms = ", ".join(cluster_top_terms[c])
    print(f"Cluster {c} (n={n:>4d}): {terms}")

### Your interpretation matters

Try giving each cluster a short label.

Then ask:

- Is it narrow or broad?
- Do two clusters appear related?
- Are any difficult to name?
- Might another value of `K` produce a more useful structure?

The algorithm has found groups.

**We still have to decide whether the groups mean anything.**

## 8) Validate the labels with representative papers

Top terms can mislead. A stronger interpretation checks actual observations.

For each cluster, we will inspect papers whose TF-IDF vectors are closest to that cluster's centroid.

In [ ]:
def closest_to_centroid(c, n=3):
    mask = (df["cluster"] == c).to_numpy()
    idx = np.where(mask)[0]

    centroid = kmeans.cluster_centers_[c].reshape(1, -1)
    sim = cosine_similarity(X[idx], centroid).ravel()
    top_local = sim.argsort()[::-1][:n]

    return idx[top_local]

for c in range(K):
    label = ", ".join(cluster_top_terms[c][:3])
    print(f"\n=== Cluster {c}: {label} ===")

    for j in closest_to_centroid(c, n=3):
        title = textwrap.shorten(
            str(df.iloc[j]["title"]),
            width=110,
            placeholder="…",
        )
        print("  •", title)

Compare the centroid terms with the representative titles.

If they tell the same story, your interpretation becomes more credible.

If they conflict, investigate before trusting the label.

> **Never trust an automatically generated cluster label without inspecting the underlying observations.**

## 9) Create a 2-D view with TruncatedSVD

TF-IDF lives in thousands of dimensions.

**TruncatedSVD** gives us a low-dimensional projection that works efficiently with sparse matrices.

We will reduce the representation to two dimensions.

### Important caution

This is **a view, not a truth**.

A 2-D projection necessarily loses information. Use the picture to inspect broad structure, not as proof that clusters are naturally separated.

In [ ]:
svd = TruncatedSVD(
    n_components=2,
    random_state=42,
)

coords = svd.fit_transform(X)

print("2-D coordinates:", coords.shape)
print(
    "Variance represented by the first two components:",
    f"{svd.explained_variance_ratio_.sum():.2%}"
)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

for c in range(K):
    mask = (df["cluster"] == c).to_numpy()

    ax.scatter(
        coords[mask, 0],
        coords[mask, 1],
        s=14,
        alpha=0.55,
        label=f"{c}: {', '.join(cluster_top_terms[c][:2])}",
    )

ax.set_title(
    f"TruncatedSVD view of TF-IDF space ({K} KMeans clusters)"
)
ax.set_xlabel("SVD component 1")
ax.set_ylabel("SVD component 2")
ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=8,
)

plt.tight_layout()
plt.show()

Look for broad separation, overlap, and outliers—but also look at the variance explained by the first two components.

If those two dimensions explain only a small share of the total variation, the plot is showing only a small slice of the full TF-IDF structure.

## 10) Cluster sizes

Cluster sizes tell us how much of **this sampled corpus** falls into each discovered group.

They do not automatically tell us which machine-learning areas are globally largest, fastest growing, or most important.

In [ ]:
sizes = df["cluster"].value_counts().sort_index()

labels = [
    f"{c}: {', '.join(cluster_top_terms[c][:2])}"
    for c in range(K)
]

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(
    range(K),
    sizes.values,
)

ax.set_yticks(range(K))
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("Number of papers in this corpus")
ax.set_title("Cluster sizes in the sampled corpus")

for bar, n in zip(bars, sizes.values):
    ax.text(
        bar.get_width() + 4,
        bar.get_y() + bar.get_height() / 2,
        str(n),
        va="center",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

## 11) Try a different value of K

A smaller `K` gives broader themes. A larger `K` gives finer distinctions.

There may not be one objectively correct answer.

Use the helper below to compare alternatives.

In [ ]:
def summarize_k(k):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10,
    )

    labels = model.fit_predict(X)

    print(f"\n=== K = {k} ===")
    print(
        "Cluster sizes:",
        pd.Series(labels).value_counts().sort_index().to_dict()
    )

    for c in range(k):
        centroid = model.cluster_centers_[c]
        top_idx = centroid.argsort()[::-1][:6]
        terms = ", ".join(feature_names[j] for j in top_idx)
        print(f"  Cluster {c}: {terms}")

# Try one or both:
# summarize_k(6)
# summarize_k(15)

## 12) Add a quantitative check: silhouette score

So far, we have evaluated cluster solutions mainly by asking whether the groups are interpretable.

We can also calculate a **silhouette score**, which compares how close each document is to its own cluster versus other clusters.

Higher values generally indicate cleaner separation.

But there is an important caveat:

> **A higher silhouette score does not automatically mean we found the "correct" number of topics.**

Clustering is unsupervised. A mathematically cleaner partition may be less useful or less interpretable for the question we care about.

So use silhouette as **one piece of evidence**, alongside:

- cluster labels;
- representative papers;
- cluster sizes;
- and substantive usefulness.

In [ ]:
for k in [5, 6, 8, 10, 12, 15, 20]:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10,
    )

    labels = model.fit_predict(X)

    score = silhouette_score(
        X,
        labels,
        metric="cosine",
    )

    print(f"K={k:2d}  silhouette={score:.3f}")

### Interpreting the result

In this corpus, the silhouette scores are likely to be **low overall**.

That is not a failure of the analysis.

It suggests that the corpus contains recognizable topical neighborhoods without breaking neatly into a small number of sharply separated categories.

That is often what real text data look like.

Now compare the numerical result with the human interpretation:

1. Which `K` gives the highest silhouette score?
2. Does that solution also produce the most interpretable clusters?
3. Does a larger `K` begin creating tiny, idiosyncratic clusters?
4. Would you choose the numerically best `K`, or the most useful `K` for your analytical purpose?

This is the larger lesson:

> **Unsupervised learning still requires judgment.**

### Questions to consider

After trying `K = 6` and `K = 15`:

1. Which themes merge when `K` becomes smaller?
2. Which themes split when `K` becomes larger?
3. Is the more detailed solution necessarily better?
4. Which value produces the most **useful** description for the question you care about?

In unsupervised learning, usefulness and interpretability often matter more than finding a mythical single "correct" partition.

## 13) TF-IDF versus embeddings

We have now analyzed the **same corpus** with two different representations.

### TF-IDF

Similarity is driven by **shared distinctive vocabulary**.

**Advantages**
- transparent;
- cheap;
- easy to inspect;
- sparse and scalable;
- features have explicit names.

**Limitation**
Two documents can express the same idea using different vocabulary and receive relatively low similarity.

### Embeddings

Similarity is based on a **learned semantic representation**.

**Advantages**
- can connect related ideas expressed with different words;
- captures more contextual and semantic information.

**Limitation**
The individual dimensions are not directly human-readable.

### The progression

\[
\text{hand-engineered features}
\rightarrow
\text{TF-IDF vocabulary features}
\rightarrow
\text{learned embeddings}
\]

Each step gives the computer more responsibility for constructing the representation.

## What we built

Starting from the fixed OpenAlex corpus, we created:

- a cleaned text field;
- a sparse TF-IDF document-term matrix;
- an inspectable lexical fingerprint;
- a cosine-similarity retrieval system;
- a KMeans cluster solution;
- cluster interpretations based on centroid terms;
- validation using representative documents;
- a TruncatedSVD visualization.

## The larger take away

Representation determines what **similar** means.

TF-IDF asks:

> **Do these documents use similar distinctive language?**

Embeddings ask:

> **Do these documents appear to mean similar things?**

Neither question is universally better. The right representation depends on what information matters for the problem you are trying to solve (as usual).